# 第12周 · 课堂练习：分词、NLL、get_batch 与因果掩码

四个小练习，对应 GPT 训练流水线上的四个零件：把字符变成整数；用 NLL 给模型打分（并学会给自己的评估代码做阳性对照）；切出训练 batch；只让模型看过去。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math

import numpy as np
import torch
from torch.nn import functional as F
from zero2researcher import data
from zero2researcher.checks import week12 as ck
from zero2researcher.checks import report

## E1 · 字符级分词：stoi、itos、encode、decode

对 Tiny Shakespeare 全文：

1. `chars`：全部不重复字符，**排好序**（`sorted`）——排序保证每次运行、每台机器的编号都一样；
2. `stoi`：字符 → 整数，`itos`：整数 → 字符；
3. `encode(s)`：字符串 → 整数列表；`decode(ids)`：整数列表 → 字符串，满足 `decode(encode(s)) == s`。

In [ ]:
text = data.load_shakespeare()
# 在这里写你的代码
raise NotImplementedError
print(len(chars), "个字符；encode('ROMEO') =", encode("ROMEO"))
ck.stoi_itos(stoi, itos)
ck.encode_decode(encode, decode)

**想一想**：`encode("Hello, 世界")` 会发生什么？字符级分词遇到训练文本里没有的字符怎么办？（一句话）

（在这里回答）

## E2 · bigram 的 NLL：平滑的代价，以及给评估代码做阳性对照

用名字数据（`data.load_baby_names()`，字符表 27，`.` = 0）。这次用**向量化**的写法代替课上的双重 for 循环：

1. `xs`、`ys`：全部 bigram 的（当前字符, 下一个字符）编号，各是一个长度 228,146 的整数张量（已给出）；
2. `counts_matrix()`：返回 27 × 27 计数矩阵——提示：`N.index_put_((xs, ys), torch.ones(len(xs)), accumulate=True)` 一行搞定；
3. `avg_nll(P)`：给定任意 27 × 27 的概率矩阵，返回平均 NLL——提示：`P[xs, ys]` 一次取出全部 228,146 个概率。

然后对平滑量 $\alpha \in \{0, 1, 10, 100\}$ 各算一次 $P = (N + \alpha) / \text{行和}$ 的 NLL，并把每个 $P$ 存进 `P_by_smoothing[alpha]`——自查除了 NLL，还要检查 $P$ 的每一行之和是不是 1（讲义里的 keepdim 广播 bug 在这份数据上 NLL 看不出来，只有行和能抓住它）。最后，**用同一个 `avg_nll` 评估一个均匀模型**（每格 1/27）：如果你的 `avg_nll` 写对了，结果必须恰好是 $\ln 27$——这就是评估代码的阳性对照。

In [ ]:
words = data.load_baby_names()
name_stoi = {ch: i + 1 for i, ch in enumerate(sorted(set("".join(words))))}
name_stoi["."] = 0                                  # 换个名字，别覆盖 E1 的 stoi（encode 还要用）
pairs = [(name_stoi[a], name_stoi[b]) for w in words for a, b in zip("." + w, w + ".")]
xs = torch.tensor([a for a, _ in pairs])            # (228146,)
ys = torch.tensor([b for _, b in pairs])            # (228146,)


def counts_matrix():
    # 在这里写你的代码
    raise NotImplementedError


def avg_nll(P):
    # 在这里写你的代码
    raise NotImplementedError


N = counts_matrix()
nll_by_smoothing, P_by_smoothing = {}, {}
for alpha in [0, 1, 10, 100]:
    # 在这里写你的代码
    raise NotImplementedError
    print(f"平滑 α = {alpha:3d}: NLL = {nll_by_smoothing[alpha]:.4f}")

uniform_nll = avg_nll(torch.full((27, 27), 1 / 27))
print(f"均匀模型: NLL = {uniform_nll:.4f}")
ck.bigram_nll(nll_by_smoothing[1], P=P_by_smoothing[1])
ck.uniform_nll(uniform_nll)

**问题**（两三句话）：平滑越大，训练集上的 NLL 越高——那为什么还要平滑？$\alpha = 0$ 在这里能算出有限的数，什么情况下它会算出无穷大？

（在这里回答）

## E3 · get_batch

写 `get_batch(data, block_size, batch_size)`：随机选 `batch_size` 个起点 `i`，`x` 取 `data[i : i + block_size]`，`y` 取 `data[i + 1 : i + block_size + 1]`，各用 `torch.stack` 叠成 `(batch_size, block_size)`。起点的范围要保证 `y` 不越界。

In [ ]:
def get_batch(data, block_size, batch_size):
    # 在这里写你的代码
    raise NotImplementedError


ck.get_batch_shapes(get_batch)
ids = torch.tensor(list(text.encode("ascii")))      # 演示用：直接拿 ASCII 码当编号（与 E1 无关）
torch.manual_seed(0)
x, y = get_batch(ids, block_size=8, batch_size=4)
print("x[0]:", repr(bytes(x[0].tolist()).decode()), "   y[0]:", repr(bytes(y[0].tolist()).decode()))

**问题**：一个 `(batch_size, block_size) = (32, 64)` 的 batch 里一共包含多少个"给定上文、预测下一个字符"的训练样本？为什么？

（在这里回答）

## E4 · 因果掩码与"对过去求平均"

1. 造出 $T = 5$ 的因果掩码 `tril`：下三角全 1、上三角全 0（`mask[i, j] = 1` 当且仅当 $j \le i$）；
2. 用它算出权重矩阵 `wei`：先把一个全 0 的 $T \times T$ 矩阵在 `tril == 0` 的位置填成 $-\infty$，再按行 softmax；
3. 验证：`wei @ x`（`x` 是 `(T, C)`）的第 t 行，等于 `x` 前 t+1 行的平均值（下面已经用 for 循环算好了 `running_mean` 供对照）。

In [ ]:
T, C = 5, 3
torch.manual_seed(0)
x = torch.randn(T, C)
running_mean = torch.stack([x[: t + 1].mean(dim=0) for t in range(T)])    # (T, C)：for 循环版

# 在这里写你的代码
raise NotImplementedError
print(wei)
ck.causal_mask(tril)
report("wei @ x 等于对过去求平均", torch.allclose(out, running_mean, atol=1e-6),
       "两者不一致", "softmax 要沿最后一维（dim=-1）；填 -inf 的位置是 tril == 0")

**问题**：真正的自注意力里，`wei` 的初值不是全 0，而是 $q \cdot k / \sqrt{d}$。这会把"均匀平均"变成什么？因果掩码这一步还需要吗？

（在这里回答）